In [17]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [18]:
import os

IMAGE_DIR = "/content/drive/MyDrive/Camscanner/Camscanner_Dataset"
files = sorted([f for f in os.listdir(IMAGE_DIR) if f.endswith(".png")])
print(f"Found {len(files)} images:")
for f in files:
    print(f)

Found 33 images:
IMG_8466 2.png
IMG_8467 2.png
IMG_8468 2.png
IMG_8469 2.png
IMG_8470 2.png
IMG_8471 2.png
IMG_8472 2.png
IMG_8473 2.png
IMG_8474 2.png
IMG_8475 2.png
IMG_8476 2.png
IMG_8477 2.png
IMG_9115.png
IMG_9116.png
IMG_9117.png
IMG_9118.png
IMG_9119.png
IMG_9120.png
IMG_9121.png
IMG_9122.png
IMG_9123.png
IMG_9330 2.png
IMG_9331 2.png
IMG_9332 2.png
IMG_9333 2.png
IMG_9334 2.png
IMG_9335 2.png
IMG_9336 2.png
IMG_9337 2.png
IMG_9338 2.png
IMG_9339 2.png
IMG_9340 2.png
IMG_9341 2.png


In [22]:
!pip install gradio

In [24]:
import gradio as gr
import cv2
import json
import os
import numpy as np

IMAGE_DIR = "/content/drive/MyDrive/Camscanner/Camscanner_Dataset"
OUTPUT_FILE = "/content/drive/MyDrive/Camscanner/Ground_Truth/ground_truth.json"

image_files = sorted([f for f in os.listdir(IMAGE_DIR) if f.lower().endswith(".png")])
ground_truth = {}
current_index = [0]
current_points = []

def load_image(index):
    fname = image_files[index]
    path = os.path.join(IMAGE_DIR, fname)
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img, fname

def click_image(img, evt: gr.SelectData):
    x, y = evt.index[0], evt.index[1]
    current_points.append((x, y))

    # Draw points on image
    img_copy = np.array(img).copy()
    labels = ["TL", "TR", "BR", "BL"]
    colors = [(255,0,0), (0,255,0), (0,0,255), (255,255,0)]

    for i, (px, py) in enumerate(current_points):
        cv2.circle(img_copy, (px, py), 15, colors[i], -1)
        cv2.putText(img_copy, labels[i], (px+10, py-10),
                   cv2.FONT_HERSHEY_SIMPLEX, 1, colors[i], 2)

    if len(current_points) == 4:
        pts = current_points + [current_points[0]]
        for i in range(4):
            cv2.line(img_copy, pts[i], pts[i+1], (0,255,0), 3)

    n = len(current_points)
    remaining = ["TL", "TR", "BR", "BL"][n:] if n < 4 else []
    status = f"Clicked {n}/4 corners. " + (f"Next: {remaining[0]}" if remaining else "✓ Done! Click Save.")

    return img_copy, status

def save_and_next():
    if len(current_points) != 4:
        return load_image(current_index[0])[0], "Need exactly 4 points before saving!", f"{current_index[0]+1}/{len(image_files)}"

    fname = image_files[current_index[0]]
    ground_truth[fname] = current_points.copy()

    # Save JSON
    with open(OUTPUT_FILE, "w") as f:
        json.dump(ground_truth, f, indent=2)

    current_points.clear()
    current_index[0] += 1

    if current_index[0] >= len(image_files):
        return None, f"✓ All done! {len(ground_truth)} images annotated.", f"{len(image_files)}/{len(image_files)}"

    img, fname = load_image(current_index[0])
    return img, f"Saved! Now annotating: {fname}", f"{current_index[0]+1}/{len(image_files)}"

def reset_points():
    current_points.clear()
    img, fname = load_image(current_index[0])
    return img, "Points reset. Click TL first.", f"{current_index[0]+1}/{len(image_files)}"

# Build UI
with gr.Blocks() as app:
    gr.Markdown("## Corner Annotator\nClick **Top-Left → Top-Right → Bottom-Right → Bottom-Left**")

    with gr.Row():
        progress = gr.Textbox(value=f"1/{len(image_files)}", label="Progress", scale=1)
        status = gr.Textbox(value="Click Top-Left corner first", label="Status", scale=3)

    img_display = gr.Image(
        value=load_image(0)[0],
        label=image_files[0],
        interactive=True
    )

    with gr.Row():
        reset_btn = gr.Button("🔄 Reset Points", variant="secondary")
        save_btn = gr.Button("✅ Save & Next Image", variant="primary")

    img_display.select(click_image, inputs=[img_display], outputs=[img_display, status])
    reset_btn.click(reset_points, outputs=[img_display, status, progress])
    save_btn.click(save_and_next, outputs=[img_display, status, progress])

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f6d3ceb743119d0923.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
